# 02 - SDPO on W&B serverless RL

[SDPO](https://arxiv.org/abs/2601.20802) trains the policy toward a *self-teacher*: the same model, re-reading its own
attempt with feedback (here, a correct sibling solution) in context. Its advantage is per token:

$$A_t = \lambda\,A^{\text{GRPO}} + (1-\lambda)\,\log\frac{\pi_\theta(y_t\mid x, f, y_{<t})}{\pi_\theta(y_t\mid x, y_{<t})}$$

The serverless API has no per-token advantage field, only one scalar reward per trajectory. `rlm_rl/sdpo.py` gets around that:

1. **Teacher logprobs.** Score each student response under the teacher context with the *current checkpoint*, using vLLM
   `prompt_logprobs` on the chat endpoint (no generation needed).
2. **Injection.** Overwrite the per-token logprobs on each `Choice` with the teacher's. ART uses those as the "old" logprobs.
3. **Advantage.** Train with `kimi_k2_tau=tau`, which makes ART compute `A_t = A_grpo - tau * (logp_new - logp_old)`,
   i.e. `A_grpo + tau * log(pi_teacher / pi_student)`. `epsilon = epsilon_high = 0` pins the importance weight to 1.
   `tau = (1 - lambda) / lambda`; the paper's lambda = 0.9 gives tau of about 0.11.

This notebook (a) checks the server really uses the injected values, then (b) runs a short SDPO-hybrid vs GRPO comparison
on `Qwen/Qwen3.6-35B-A3B`. Part (b) is a few steps only, so treat its numbers as a smoke test, not a result.

In [1]:
import copy, os, sys
from pathlib import Path

from dotenv import load_dotenv

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")  # provides WANDB_API_KEY
assert os.environ.get("WANDB_API_KEY"), "WANDB_API_KEY missing from .env"

import art
from art.serverless.backend import ServerlessBackend

from rlm_rl.tasks import make_tasks
from rlm_rl.rollout import evaluate, gather_groups
from rlm_rl.sdpo import inject_teacher_logprobs, train_kwargs

In [2]:
BASE_MODEL = "Qwen/Qwen3.6-35B-A3B"
PROJECT = "rlm-sdpo"
RUN_TAG = "v1"
LAMBDA = 0.9
TAU = (1 - LAMBDA) / LAMBDA
LEARNING_RATE = 1e-5
TASKS_PER_STEP = 8
ROLLOUTS_PER_TASK = 4
COMPARE_STEPS = 3
EVAL_TASKS = make_tasks(16, seed=999)
COMMON = {"epsilon": 0.0, "epsilon_high": 0.0}  # identical IS settings for every arm

backend = ServerlessBackend()

async def new_model(arm: str) -> art.TrainableModel:
    model = art.TrainableModel(
        name=f"qwen3-6-35b-a3b-{arm}", project=PROJECT,
        run_name=f"qwen3-6-35b-a3b-{arm}-{RUN_TAG}", base_model=BASE_MODEL,
    )
    await model.register(backend)
    return model

## (a) Does the server use the injected teacher logprobs?

Three fresh LoRAs (all identical to the base model at step 0) train on the **same** rollouts:

- `grpo-a` and `grpo-b`: plain GRPO. Their gap is the run-to-run noise (random LoRA init).
- `sdpo-check`: same rollouts with teacher logprobs injected and `kimi_k2_tau=τ`.

If the server ignored our logprobs or τ, `sdpo-check` would look like the GRPO pair. A gap well beyond the GRPO-A/B
noise means the SDPO advantages reached the loss.

In [3]:
wiring = {arm: await new_model(arm) for arm in ["grpo-a", "grpo-b", "sdpo-check"]}
groups = await gather_groups(wiring["grpo-a"], 0, make_tasks(TASKS_PER_STEP, seed=100), ROLLOUTS_PER_TASK)
sdpo_groups, stats = await inject_teacher_logprobs(wiring["sdpo-check"], 0, groups)
print("rewards per group:", [[t.reward for t in g.trajectories] for g in groups])
print("teacher re-scoring:", stats.summary())

rollouts@step0:   0%|          | 0/32 [00:00<?, ?it/s]

rewards per group: [[1.0, 1.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 0.0], [1.0, 1.0, 0.0, 0.0], [1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0]]
teacher re-scoring: {'rescored': 27, 'skipped_no_solution': 4, 'skipped_misaligned': 1, 'mean_token_log_ratio': -0.054869541269646604, 'min_token_log_ratio': -13.306263476610184}


In [4]:
check = {}
check["grpo-a"] = await backend.train(wiring["grpo-a"], copy.deepcopy(groups), learning_rate=LEARNING_RATE, **COMMON)
check["grpo-b"] = await backend.train(wiring["grpo-b"], copy.deepcopy(groups), learning_rate=LEARNING_RATE, **COMMON)
check["sdpo-check"] = await backend.train(wiring["sdpo-check"], sdpo_groups, learning_rate=LEARNING_RATE, **train_kwargs(TAU))

keys = sorted({k for r in check.values() for k in r.metrics if k.startswith("loss/")})
print(f"{'metric':28}" + "".join(f"{arm:>14}" for arm in check))
for k in keys:
    print(f"{k:28}" + "".join(f"{check[arm].metrics.get(k, float('nan')):>14.5f}" for arm in check))

train:   0%|          | 0/9 [00:00<?, ?it/s]

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


wandb: Currently logged in as: rishi-jain1 (models-ontario-tech-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Tracking run with wandb version 0.30.0


wandb: Run data is saved locally in F:\Personal\RLM\RLM\notebooks\wandb\run-20261010_015426-qwen3-6-35b-a3b-grpo-a-v1
wandb: Run `wandb offline` to turn off syncing.


...

wandb: Resuming run qwen3-6-35b-a3b-sdpo-check-v1


wandb:  View project at https://wandb.ai/models-ontario-tech-university/rlm-sdpo


wandb:  View run at https://wandb.ai/models-ontario-tech-university/rlm-sdpo/runs/qwen3-6-35b-a3b-sdpo-check-v1


wandb: Initializing weave.


metric                              grpo-a        grpo-b    sdpo-check
loss/clipped_token_fraction        0.90149       0.90276       0.96208
loss/grad_norm                     1.61892       1.60814       1.61789
loss/importance_ratio_mean         1.00030       1.00040      20.53216
loss/importance_ratio_p95          1.06659       1.06632       1.16747
loss/importance_ratio_p99          1.25664       1.25744       4.73832
loss/probs_corr                    0.97754       0.97829       0.87315
loss/train                        -0.00762      -0.00809      -0.00980


## (b) Short SDPO-hybrid vs GRPO run

Two fresh LoRAs, each sampling its own rollouts every step. The SDPO arm re-scores its rollouts with its own current
checkpoint as teacher before training.

In [5]:
arms = {"grpo": await new_model("grpo"), "sdpo": await new_model("sdpo")}
curves = {arm: [{"step": 0, "eval_acc": await evaluate(m, 0, EVAL_TASKS)}] for arm, m in arms.items()}

for arm, model in arms.items():
    step = 0
    for i in range(COMPARE_STEPS):
        groups = await gather_groups(model, step, make_tasks(TASKS_PER_STEP, seed=200 + i), ROLLOUTS_PER_TASK)
        await model.log(groups, split="train", step=step)
        row = {"train_reward": sum(t.reward for g in groups for t in g.trajectories) / sum(len(g.trajectories) for g in groups)}
        if arm == "sdpo":
            groups, stats = await inject_teacher_logprobs(model, step, groups, seed=i)
            row.update(stats.summary())
            result = await backend.train(model, groups, learning_rate=LEARNING_RATE, **train_kwargs(TAU))
        else:
            result = await backend.train(model, groups, learning_rate=LEARNING_RATE, **COMMON)
        await model.log(metrics=result.metrics, step=result.step)
        step = result.step
        row.update(step=step, loss=result.metrics.get("loss/train"))
        curves[arm].append(row)
    curves[arm][-1]["eval_acc"] = await evaluate(model, step, EVAL_TASKS)

rollouts@step0:   0%|          | 0/32 [00:00<?, ?it/s]

wandb: setting up run qwen3-6-35b-a3b-grpo-v1


wandb: Tracking run with wandb version 0.30.0


wandb: Run data is saved locally in F:\Personal\RLM\RLM\notebooks\wandb\run-20261010_020218-qwen3-6-35b-a3b-grpo-v1
wandb: Run `wandb offline` to turn off syncing.


wandb: Resuming run qwen3-6-35b-a3b-grpo-v1


...

weave: 🍩 https://wandb.ai/models-ontario-tech-university/rlm-sdpo/r/call/01a12474-1527-7de1-9672-31e17eef0d5a


weave: 🍩 https://wandb.ai/models-ontario-tech-university/rlm-sdpo/r/call/01a12474-14dc-7e6e-875a-ef2452146e52


weave: 🍩 https://wandb.ai/models-ontario-tech-university/rlm-sdpo/r/call/01a12474-15a0-720a-a8b5-880eab178049


weave: 🍩 https://wandb.ai/models-ontario-tech-university/rlm-sdpo/r/call/01a12474-159e-729f-9c27-48c81fc32268


train:   0%|          | 0/13 [00:00<?, ?it/s]

In [6]:
for arm, rows in curves.items():
    print(arm)
    for row in rows:
        print("   ", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()})

grpo
    {'step': 0, 'eval_acc': 0.75}
    {'train_reward': 0.5938, 'step': 1, 'loss': -0.0131}
    {'train_reward': 0.875, 'step': 2, 'loss': -0.0124}
    {'train_reward': 0.7812, 'step': 3, 'loss': -0.0031, 'eval_acc': 0.8438}
sdpo
    {'step': 0, 'eval_acc': 0.75}
    {'train_reward': 0.7812, 'rescored': 29, 'skipped_no_solution': 1, 'skipped_misaligned': 2, 'mean_token_log_ratio': -0.053, 'min_token_log_ratio': -13.7311, 'step': 1, 'loss': -0.0061}
    {'train_reward': 0.7812, 'rescored': 31, 'skipped_no_solution': 1, 'skipped_misaligned': 0, 'mean_token_log_ratio': -0.0499, 'min_token_log_ratio': -17.367, 'step': 2, 'loss': -0.0197}
    {'train_reward': 0.8438, 'rescored': 31, 'skipped_no_solution': 0, 'skipped_misaligned': 1, 'mean_token_log_ratio': -0.0559, 'min_token_log_ratio': -17.9189, 'step': 3, 'loss': -0.0019, 'eval_acc': 0.8125}
